In [15]:
import json
import csv
import numpy as np
import random
import datetime

# Classification task

In [63]:
csv_file = open("../../emoticon-data/classification.csv", "r")
reader = csv.DictReader(csv_file, delimiter=",")
data = [row for row in reader]
len(data)

211247

In [64]:
dims = ["anger","contempt","disgust","fear","happiness","neutral","sadness","surprise"]
cur_stat = {dim : 0.0 for dim in dims}
result = []

In [65]:
while len(result) < 20:
    sel = random.randint(1, len(data))
    new_stat = { dim: cur_stat[dim] + float(data[sel][dim]) for dim in dims}
    new_min = np.min([a for a in new_stat.values()])
    new_max = np.max([a for a in new_stat.values()])
    if new_max - new_min <= 1.0:
        cur_stat = new_stat
        result.append(data[sel]["file"])

In [66]:
for f in result:
    print(f)

img_align_celeba/132037.jpg
lfw/George_W_Bush/George_W_Bush_0358.jpg
img_align_celeba/201003.jpg
img_align_celeba/170268.jpg
img_align_celeba/133854.jpg
img_align_celeba/200106.jpg
img_align_celeba/033137.jpg
img_align_celeba/017604.jpg
img_align_celeba/195145.jpg
img_align_celeba/043637.jpg
img_align_celeba/066661.jpg
img_align_celeba/025139.jpg
img_align_celeba/149909.jpg
img_align_celeba/096211.jpg
img_align_celeba/153904.jpg
lfw/Lindsay_Davenport/Lindsay_Davenport_0007.jpg
img_align_celeba/071303.jpg
lfw/Monica_Seles/Monica_Seles_0001.jpg
img_align_celeba/033137.jpg
img_align_celeba/060966.jpg


In [67]:
cur_stat

{'anger': 2.6299999999999994,
 'contempt': 2.155,
 'disgust': 2.8989999999999996,
 'fear': 1.9729999999999996,
 'happiness': 2.538,
 'neutral': 2.5709999999999997,
 'sadness': 2.7199999999999993,
 'surprise': 2.5039999999999996}

In [68]:
id = "encode-"+datetime.datetime.now().isoformat()
print(f"Writing task definition for {id}")
with open("gen/work.json", "w") as file:
    json.dump({
        "type": "classify",
        "id": id,
        "data": [{"file": file for file in result}]
    }, file, indent=2)

Writing task definition for encode-2022-01-12T12:31:58.583033


# Recognition task

In [70]:
resultA = json.load(open('gen/resultA.json', 'r'))
resultB = json.load(open('gen/resultB.json', 'r'))

In [71]:
with open("gen/selection.csv","r") as file:
    selected_faces = [row['file'] for row in csv.DictReader(file)]
selected_faces.sort()

In [72]:
def convert_emoji_params(x):
    a1 = x[1] * 100
    a2 = x[2] * 100
    return {
            "valence": round(100 * x[0]),
            "arousal": round((a1+a2)/2),
            "potency": round(100 * x[3]),
            "contempt": round(100 * x[4]),
            "expression": round((a2-a1)/2 + 50)
        }

In [73]:
data = []
for key in selected_faces:
    others = [{"file":face} for face in selected_faces if face!=key]
    data.append({
        "file": key,
        "B": convert_emoji_params(resultB[key]["x_opt"]),
        "A":  {
            "code": max(resultA[key]["x_opt"].items(), key=lambda x: x[1])[0]
        },
        "decoy_sets": [
            np.random.permutation(
                np.random.choice(others, 5, replace=False).tolist() + [{"file":key}]
             ).tolist()
            for _ in range(5)
        ]
    })

In [74]:
id = "decode-"+datetime.datetime.now().isoformat()
print(f"Writing task definition for {id}")
with open("gen/work.json", "w") as file:
    json.dump({
        "type": "recognize",
        "id": id,
        "data": data
    }, file, indent=2)

Writing task definition for decode-2022-01-12T12:37:24.512149
